<a href="https://colab.research.google.com/github/AnjanPayra/MM-CCNB/blob/main/MM_CCNB_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import warnings
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, mean_squared_error, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report
)

warnings.filterwarnings("ignore")
np.random.seed(42)

UPLOAD_DIR = "/content"
OUT_DIR = "/mnt/user-data/outputs"
os.makedirs(OUT_DIR, exist_ok=True)

DATASETS = ["YDIP", "YHQ", "YMBD", "YMIPS"]
K_SIGMA = 1  # K in the 3-point (K-sigma) threshold rule


# ======================================================================
# 0. Reference tables
# ======================================================================
def load_essential_set():
    e = pd.read_excel(os.path.join(UPLOAD_DIR, "Essential.xlsx"), header=None)
    return set(e[0].astype(str).str.strip())


COMPARTMENT_KEYWORDS = [
    "nucleus", "nucleolus", "cytoplasm", "cytosol", "mitochondri",
    "golgi", "endoplasmic reticulum", "vacuole", "bud", "cell membrane",
    "membrane", "peroxisome", "cell wall", "ribosome", "endosome",
    "chromosome", "spindle", "nuclear", "secreted", "lipid particle"
]

def load_subcellular_map():
    sc = pd.read_csv(os.path.join(UPLOAD_DIR, "Sub_cellular.csv"))
    sc.columns = ["gene", "location_text"]
    sc = sc.dropna(subset=["gene"])
    sl_map = {}
    for _, row in sc.iterrows():
        gene = str(row["gene"]).strip()
        text = str(row["location_text"]).lower()
        found = {kw for kw in COMPARTMENT_KEYWORDS if kw in text}
        if found:
            sl_map[gene] = found
    return sl_map


def load_go_scores(dataset):
    """Precomputed protein-wise GO_Nb(u) values."""
    path = os.path.join(UPLOAD_DIR, f"{dataset}_GO_final.csv")
    go = pd.read_csv(path, header=None, names=["protein", "GO_Nb"])
    go["protein"] = go["protein"].astype(str).str.strip()
    return dict(zip(go["protein"], go["GO_Nb"]))


# ======================================================================
# 1. Network
# ======================================================================
def load_network(dataset):
    path = os.path.join(UPLOAD_DIR, f"{dataset}.txt")
    df = pd.read_csv(path)
    df.columns = ["p1", "p2"]
    df = df.dropna()
    df = df[df["p1"].astype(str).str.strip() != df["p2"].astype(str).str.strip()]
    G = nx.Graph()
    G.add_edges_from(zip(df["p1"].astype(str).str.strip(), df["p2"].astype(str).str.strip()))
    G.remove_edges_from(nx.selfloop_edges(G))
    return G


# ======================================================================
# 2.1 -> 1.2  Edge-Clustering-Coefficient, protein-wise ECC(u)
# ======================================================================
def compute_ecc(G):
    """
    ECC(u,v) = (# triangles containing edge u-v) / max(deg(u)-1, deg(v)-1)
    ECC(u)   = sum over level-1 neighbours t of ECC(u,t)
    """
    ecc_edge = {}
    deg = dict(G.degree())
    for u, v in G.edges():
        common = len(list(nx.common_neighbors(G, u, v)))
        denom = max(deg[u] - 1, deg[v] - 1)
        ecc_edge[(u, v)] = common / denom if denom > 0 else 0.0

    ecc_node = {n: 0.0 for n in G.nodes()}
    for (u, v), val in ecc_edge.items():
        ecc_node[u] += val
        ecc_node[v] += val
    return ecc_node


# ======================================================================
# 3.1 -> 3.2  Localized significance score, protein-wise SL_Nb(u)
# ======================================================================
def compute_sl_nb(G, sl_map):
    """
    SL_Nb(u,v) = |Union(SL(t)) for t in common_neigh(u,v)|^2 / (|SL(u)|*|SL(v)|)
    SL_Nb(u)   = sum over level-1 neighbours t of SL_Nb(u,t)
    """
    sl_edge = {}
    for u, v in G.edges():
        su, sv = sl_map.get(u), sl_map.get(v)
        if su and sv:
            common = list(nx.common_neighbors(G, u, v))
            union_locs = set()
            for t in common:
                union_locs |= sl_map.get(t, set())
            r = len(union_locs) ** 2
            s = len(su) * len(sv)
            sl_edge[(u, v)] = r / s if s > 0 else 0.0
        else:
            sl_edge[(u, v)] = 0.0

    sl_node = {n: 0.0 for n in G.nodes()}
    for (u, v), val in sl_edge.items():
        sl_node[u] += val
        sl_node[v] += val
    return sl_node


# ======================================================================
# 4. Feature Essentiality Score + MAX-MIN strategy (Modified Jaccard)
# ======================================================================
def compute_es(row):
    """ES(u) = Jaccard_mod(u) = MIN(ECC(u), MAX(GO_Nb(u), SL_Nb(u)))"""
    return min(row["ECC"], max(row["GO_Nb"], row["SL_Nb"]))


# ======================================================================
# 3-point / K-sigma threshold (Fig.9's "3-point Threshold" table)
# ======================================================================
def k_sigma_threshold(values, K=K_SIGMA):
    values = np.asarray(values, dtype=float)
    alpha = values.mean()
    sigma = values.std()
    var = values.var()
    return alpha + K * sigma * (1 - 1 / (1 + var ** 2))


# ======================================================================
# Build the per-protein feature/score table for one dataset
# ======================================================================
def build_dataset(dataset, essential_set, sl_map):
    print(f"\n[{dataset}] loading network ...")
    G = load_network(dataset)
    print(f"  nodes={G.number_of_nodes()}  edges={G.number_of_edges()}")

    print(f"[{dataset}] computing ECC(u) ...")
    ecc = compute_ecc(G)
    print(f"[{dataset}] computing SL_Nb(u) ...")
    sl_nb = compute_sl_nb(G, sl_map)
    go_scores = load_go_scores(dataset)

    rows = []
    for n in G.nodes():
        rows.append({
            "protein": n,
            "ECC": ecc.get(n, 0.0),
            "GO_Nb": go_scores.get(n, 0.0),
            "SL_Nb": sl_nb.get(n, 0.0),
            "essential": 1 if n in essential_set else 0,
        })
    df = pd.DataFrame(rows)
    df["ES"] = df.apply(compute_es, axis=1)  # Modified-Jaccard Essentiality Score
    return df


# ======================================================================
# Evaluation
# ======================================================================
def evaluate(df, dataset_name):
    y_true = df["essential"].values
    score = df["ES"].values

    fpr, tpr, roc_thresh = roc_curve(y_true, score)
    roc_auc = auc(fpr, tpr)
    prec, rec, _ = precision_recall_curve(y_true, score)
    ap = average_precision_score(y_true, score)

    # (a) paper's 3-point / K-sigma threshold
    thresh_paper = k_sigma_threshold(score, K=K_SIGMA)
    y_pred_paper = (score >= thresh_paper).astype(int)
    acc_paper = accuracy_score(y_true, y_pred_paper)
    mse_paper = mean_squared_error(y_true, y_pred_paper)
    cm_paper = confusion_matrix(y_true, y_pred_paper)

    # (b) data-driven Youden-optimal threshold (max TPR-FPR on ROC curve)
    youden = tpr - fpr
    best_idx = int(np.argmax(youden))
    thresh_opt = roc_thresh[best_idx]
    y_pred_opt = (score >= thresh_opt).astype(int)
    acc_opt = accuracy_score(y_true, y_pred_opt)
    mse_opt = mean_squared_error(y_true, y_pred_opt)
    cm_opt = confusion_matrix(y_true, y_pred_opt)
    report_opt = classification_report(y_true, y_pred_opt, target_names=["non-essential", "essential"])

    print(f"\n[{dataset_name}] MM-CCNB (Modified-Jaccard ES) evaluation")
    print(f"  ROC-AUC  : {roc_auc:.4f}   PR-AUC : {ap:.4f}")
    print(f"  (a) Paper 3-sigma threshold = {thresh_paper:.4f} -> Accuracy={acc_paper:.4f}  MSE={mse_paper:.4f}")
    print(f"      Confusion matrix:\n{cm_paper}")
    print(f"  (b) Youden-optimal threshold = {thresh_opt:.4f} -> Accuracy={acc_opt:.4f}  MSE={mse_opt:.4f}")
    print(f"      Confusion matrix:\n{cm_opt}")
    print(report_opt)

    return {
        "dataset": dataset_name,
        "threshold_paper": thresh_paper, "accuracy_paper": acc_paper, "mse_paper": mse_paper, "cm_paper": cm_paper,
        "threshold_opt": thresh_opt, "accuracy_opt": acc_opt, "mse_opt": mse_opt, "cm_opt": cm_opt,
        "report_opt": report_opt,
        "roc_auc": roc_auc, "pr_auc": ap,
        "fpr": fpr, "tpr": tpr, "precision": prec, "recall": rec,
    }


# ======================================================================
# Main driver
# ======================================================================
def main():
    essential_set = load_essential_set()
    sl_map = load_subcellular_map()
    print(f"Essential reference proteins : {len(essential_set)}")
    print(f"Proteins with subcellular annotation : {len(sl_map)}")

    all_tables, all_results = {}, {}

    for ds in DATASETS:
        df = build_dataset(ds, essential_set, sl_map)
        all_tables[ds] = df
        df.to_csv(os.path.join(OUT_DIR, f"{ds}_MMCCNB_feature_table.csv"), index=False)
        all_results[ds] = evaluate(df, ds)

    summary = pd.DataFrame([{
        "Dataset": ds,
        "Nodes": len(all_tables[ds]),
        "Essential(labelled)": int(all_tables[ds]["essential"].sum()),
        "Threshold_paper(3sigma)": all_results[ds]["threshold_paper"],
        "Accuracy_paper": all_results[ds]["accuracy_paper"],
        "MSE_paper": all_results[ds]["mse_paper"],
        "Threshold_Youden": all_results[ds]["threshold_opt"],
        "Accuracy_Youden": all_results[ds]["accuracy_opt"],
        "MSE_Youden": all_results[ds]["mse_opt"],
        "ROC_AUC": all_results[ds]["roc_auc"],
        "PR_AUC": all_results[ds]["pr_auc"],
    } for ds in DATASETS])
    summary.to_csv(os.path.join(OUT_DIR, "MMCCNB_summary_metrics.csv"), index=False)
    print("\n=== SUMMARY (MM-CCNB) ===")
    print(summary.to_string(index=False))

    # ---- ROC curve ----
    plt.figure(figsize=(7, 6))
    for ds in DATASETS:
        r = all_results[ds]
        plt.plot(r["fpr"], r["tpr"], label=f"{ds} (AUC={r['roc_auc']:.3f})", linewidth=2)
    plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve — Essential Protein Prediction (MM-CCNB)")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "MMCCNB_ROC_curves.png"), dpi=150)
    plt.close()

    # ---- PR curve ----
    plt.figure(figsize=(7, 6))
    for ds in DATASETS:
        r = all_results[ds]
        plt.plot(r["recall"], r["precision"], label=f"{ds} (AP={r['pr_auc']:.3f})", linewidth=2)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve — Essential Protein Prediction (MM-CCNB)")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "MMCCNB_PR_curves.png"), dpi=150)
    plt.close()

    # ---- Bar chart ----
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(DATASETS))
    width = 0.25
    ax.bar(x - width, summary["Accuracy_Youden"], width, label="Accuracy (Youden thresh.)")
    ax.bar(x, summary["ROC_AUC"], width, label="ROC-AUC")
    ax.bar(x + width, summary["PR_AUC"], width, label="PR-AUC")
    ax.set_xticks(x)
    ax.set_xticklabels(DATASETS)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score")
    ax.set_title("MM-CCNB Model Performance Across Datasets")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "MMCCNB_performance_comparison.png"), dpi=150)
    plt.close()

    print(f"\nAll outputs saved to {OUT_DIR}")
    return all_tables, all_results, summary


if __name__ == "__main__":
    main()

Essential reference proteins : 1285
Proteins with subcellular annotation : 4081

[YDIP] loading network ...
  nodes=5093  edges=24743
[YDIP] computing ECC(u) ...
[YDIP] computing SL_Nb(u) ...

[YDIP] MM-CCNB (Modified-Jaccard ES) evaluation
  ROC-AUC  : 0.6949   PR-AUC : 0.4151
  (a) Paper 3-sigma threshold = 1.5178 -> Accuracy=0.7769  MSE=0.2231
      Confusion matrix:
[[3653  273]
 [ 863  304]]
  (b) Youden-optimal threshold = 0.1635 -> Accuracy=0.6994  MSE=0.3006
      Confusion matrix:
[[2848 1078]
 [ 453  714]]
               precision    recall  f1-score   support

non-essential       0.86      0.73      0.79      3926
    essential       0.40      0.61      0.48      1167

     accuracy                           0.70      5093
    macro avg       0.63      0.67      0.64      5093
 weighted avg       0.76      0.70      0.72      5093


[YHQ] loading network ...
  nodes=4683  edges=22665
[YHQ] computing ECC(u) ...
[YHQ] computing SL_Nb(u) ...

[YHQ] MM-CCNB (Modified-Jaccard ES)